# Sentiment-Assisted Bot Detection — Colab Run

This notebook clones the project, installs dependencies, extracts the Cresci-2017
dataset, and runs the full pipeline (Feature Set A vs. Feature Set B, 3 classifiers,
McNemar's test, feature importance) on Colab's fast local disk instead of a slow
local machine.

**Before running this notebook:** upload the `cresci-2017.csv/datasets_full.csv/`
folder (the `.zip` files, e.g. `genuine_accounts.csv.zip`, `social_spambots_1.csv.zip`, ...)
to your Google Drive, e.g. under `My Drive/cresci-2017/datasets_full.csv/`.
Uploading to Drive first (rather than Colab's file-upload widget) is much more
reliable for files this large.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone the project and install dependencies

In [ ]:
!git clone https://github.com/SonponYe/project_86_-sentiment-bot-detection.git
%cd project_86_-sentiment-bot-detection
!pip install -q vaderSentiment textblob imbalanced-learn statsmodels

(pandas, numpy, scikit-learn, matplotlib, and seaborn already ship with Colab, so
only the extra packages need installing.)

## 3. Extract Cresci-2017 into data/raw/

Adjust `DRIVE_DATA_DIR` below if you uploaded the zips to a different Drive path.

This uses genuine_accounts + all social_spambots + all traditional_spambots
(matching the proposal's stated "3 bot types" composition). To also test
`fake_followers.csv.zip` as an extra bot class, add it to the `SUBSETS` list.

In [ ]:
import pathlib
import zipfile

DRIVE_DATA_DIR = pathlib.Path('/content/drive/MyDrive/cresci-2017/datasets_full.csv')
RAW_DIR = pathlib.Path('data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

SUBSETS = [
    'genuine_accounts.csv.zip',
    'social_spambots_1.csv.zip',
    'social_spambots_2.csv.zip',
    'social_spambots_3.csv.zip',
    'traditional_spambots_1.csv.zip',
    'traditional_spambots_2.csv.zip',
    'traditional_spambots_3.csv.zip',
    'traditional_spambots_4.csv.zip',
]

for name in SUBSETS:
    zip_path = DRIVE_DATA_DIR / name
    print('Extracting', zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(RAW_DIR)

print(sorted(p.name for p in RAW_DIR.iterdir()))

## 4. Run the pipeline

This loads and cleans the accounts, builds Feature Set A (metadata) and
Feature Set B (metadata + sentiment), takes one shared stratified 80/20 split,
trains Logistic Regression / Random Forest / SVM on each feature set (6
experiments), and writes results to `results/tables/`.

In [ ]:
!python -m src.pipeline

## 5. Inspect results

In [ ]:
import pandas as pd

summary = pd.read_csv('results/tables/experiment_summary.csv')
summary

In [ ]:
mcnemar = pd.read_csv('results/tables/mcnemar_significance.csv')
mcnemar

In [ ]:
importance = pd.read_csv('results/tables/feature_importance.csv')
importance.head(10)

## 6. Plot: F1 by model, metadata-only vs. metadata+sentiment

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7, 4))
sns.barplot(data=summary, x='model', y='f1', hue='feature_set')
plt.title('F1-score by model and feature set')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

## 7. Save results back to Drive (optional)

Colab's filesystem is ephemeral — copy the results out before the runtime
disconnects if you want to keep them.

In [ ]:
!mkdir -p /content/drive/MyDrive/bot_detection_results
!cp -r results/tables /content/drive/MyDrive/bot_detection_results/